# Session 2.1 : Data Transformation

_Analytics Through Coding Autumn 2026_

---

Visualisation and statistical analysis are only useful if the data are in the form required to answer the question.

In practice, we often need to:

* keep only relevant observations;
* reorder observations;
* select or rename variables;
* create new variables;
* group observations;
* calculate summaries.

In this session we will use the `flights` dataset, which contains flights that departed from New York City in 2013.

Rather than learning pandas functions in isolation, we will start with an **analytical question** and then decide what transformation is needed.

---

## Starting out

As always, import the necessary libraries and dataset at the start of the notebook.

In [6]:
import pandas as pd
import numpy as np

In [7]:
import pandas as pd
import numpy as np

flights = pd.read_csv("../Data/nycflights13_flights.csv", index_col=0)
flights.reset_index(drop=True, inplace=True)

flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00


Before transforming a dataset, remind yourself what one row represents and check the variables available.

In [8]:
print("Shape:", flights.shape)
print(flights.columns.tolist())

Shape: (202066, 19)
['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'time_hour']


## A small set of transformation tools

We will focus on a small number of pandas methods that can be combined to answer many analytical questions.

| pandas method | What it helps us do |
|---|---|
| `query()` | Keep observations that satisfy a condition |
| `sort_values()` | Reorder observations |
| `loc[]` | Select rows and/or columns |
| `rename()` | Rename variables |
| `assign()` | Create new variables |
| `groupby()` | Divide observations into groups |
| `agg()` | Calculate summaries |

The important skill is not memorising this table. It is recognising **which operation is required by the analytical question**.

## Question 1: Which flights departed on 16 August?

We only want observations where:

* `month == 8`
* `day == 16`

We can use `.query()` to filter observations.

Notice that pandas returns a **new DataFrame**. The original `flights` DataFrame has not been changed.

Multiple arguments to `.query()` are combined with `“and”`: every expression must be true in order for a row to be included in the output. For some operations you may need other Boolean operations - `&` is “and”, `|` is “or”, and `!` is “not”

![GitHub Codespaces](Boolean_operators.png)

### Exercise 1

Find all flights that:

* departed from `JFK`;
* travelled to `LAX`; and
* had a departure delay greater than 60 minutes.

Keep the result in a DataFrame called `jfk_lax_delayed`.

How many flights meet all three conditions?

## Question 2: Which flights experienced the largest departure delays?

Filtering determines **which observations we keep**.

Sorting determines **the order in which we inspect them**.

We can also sort using more than one variable.

For example, the following sorts chronologically by month and day.

### Exercise 2

Find the **10 flights with the longest arrival delays**.

Display only:

* `origin`
* `dest`
* `carrier`
* `arr_delay`

Sort the result from the largest delay to the smallest.

## Question 3: Which variables do we actually need?

Real datasets often contain many more variables than are required for a particular question.

For an analysis of flight delays, we might only need a subset of columns.

We can also rename variables when a clearer name would make later code easier to read.

<div class="alert alert-warning">
<b>Note.</b>
Selecting or renaming columns does not improve the analysis by itself. Do it when it makes the dataset easier to understand or when only a smaller set of variables is needed for the question.
</div>

## Question 4: Did flights make up time while in the air?

Sometimes the variable we need does not exist in the original dataset.

We can create a new variable from existing variables using `.assign()`.

Define:

`gain = arrival delay - departure delay`

A negative value means the flight arrived with **less delay** than it had when it departed.

We can create several new variables at the same time.

For example, approximate average speed in miles per hour can be calculated from `distance` and `air_time`.

### Exercise 3

Create a DataFrame called `flights_delay_change` containing a new variable:

`delay_change = arr_delay - dep_delay`

Then keep only flights where `delay_change <= -30`.

These are flights that reduced their delay by at least 30 minutes between departure and arrival.

Display the 10 flights with the largest reduction in delay.

## Question 5: What is a typical delay?

`.agg()` allows us to collapse many observations into summary statistics.

Without grouping, the summary describes the **entire dataset**.

## Question 6: Does delay differ between airports or airlines?

Usually we want summaries **within groups**.

`groupby()` changes the unit of analysis.

Instead of one row representing one flight, the resulting table can have one row representing one airport, airline, month, destination, or another group.

This is an important conceptual change:

> Before `groupby()`: one row = one flight  
> After `groupby()` + `agg()`: one row = one origin airport

Always know what **one row represents** after a transformation.

### Exercise 4

Calculate the following for each airline (`carrier`):

* `n_flights`: number of flights;
* `avg_dep_delay`: average departure delay;
* `avg_arr_delay`: average arrival delay.

Keep only airlines with at least 1,000 flights and sort them from the **lowest to highest average arrival delay**.

## Combining transformations

Real analytical questions usually require more than one operation.

Method chaining lets us read the analysis as a sequence:

1. start with the data;
2. create or modify variables;
3. group observations;
4. calculate summaries;
5. filter the summaries;
6. order the result.

This mirrors the analytical workflow more closely than learning each method separately.

### Final Exercise — Average speed by destination

Imagine you want to know how fast flights travel on average depending on destination.

Create a new variable:

`speed = distance / (air_time / 60)`

Then, for each destination (`dest`), calculate:

* `count`: number of flights;
* `avg_speed`: average speed in miles per hour.

Keep only destinations with **more than 20 flights** and sort them by `avg_speed` from fastest to slowest.

Display the 10 fastest destinations.

Try to write this as a single method chain.

## Check the result

A transformation is not finished just because the code ran.

Before using the result, ask:

* Does one row now represent what I think it represents?
* Are the number of groups plausible?
* Are the summary values plausible?
* Did missing values affect the calculation?
* Did filtering happen before or after aggregation as intended?

A few simple checks can prevent incorrect conclusions.

## All Done!

In this session we used pandas transformations to answer analytical questions.

We practised:

* filtering observations with `query()`;
* sorting observations with `sort_values()`;
* selecting variables with `loc[]`;
* renaming variables;
* creating new variables with `assign()`;
* grouping observations with `groupby()`;
* calculating summaries with `agg()`;
* combining several operations using method chaining;
* checking that transformed results still make analytical sense.

The key idea is:

> **Start with the question, then decide what data transformation is required.**

We will now move on to the structure of datasets and the principles of **tidy data**.